In [0]:
# # Reset Bronze — run this before anything else
# spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")
# dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
# dbutils.fs.rm(SCHEMA_LOC, recurse=True)
# print("Bronze reset complete. Ready for clean run.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

CATALOG = 'weather_project'
SCHEMA = 'default'

RAW_PATH= f'/Volumes/{CATALOG}/{SCHEMA}/raw'
REFERENCE_PATH=f'/Volumes/{CATALOG}/{SCHEMA}/reference'
CHECKPOINT_PATH= f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/bronze"
SCHEMA_LOC = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/bronze_schema"


BRONZE_TABLE     = f"{CATALOG}.{SCHEMA}.bronze_weather_raw"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_weather_quarantine"
STATION_TABLE    = f"{CATALOG}.{SCHEMA}.station_master"


In [0]:
CATALOG = 'weather_project'
SCHEMA = 'default'

REFERENCE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/reference"
STATION_TABLE  = f"{CATALOG}.{SCHEMA}.station_master"

print(f"Reference path : {REFERENCE_PATH}")
print(f"Station table  : {STATION_TABLE}")

Reference path : /Volumes/weather_project/default/reference
Station table  : weather_project.default.station_master


In [0]:
df_station= spark.read.csv(REFERENCE_PATH,header=True,inferSchema=True)
print(f'Row Counts = {df_station.count()}')
print(f'Coulmns : {df_station.columns}')

#writing it as delta table

df_station.write\
    .format('delta')\
        .mode('overwrite')\
            .saveAsTable(STATION_TABLE)

print(f"station_master saved as Delta table: {STATION_TABLE}")
#merge statement instead

Row Counts = 12
Coulmns : ['station_id', 'city', 'station_zone', 'is_active', 'commissioned']
station_master saved as Delta table: weather_project.default.station_master


In [0]:
%sql
DROP TABLE IF EXISTS weather_project.default.bronze_weather_raw;

CREATE TABLE weather_project.default.bronze_weather_raw (
  city STRING,
  country STRING,
  event_time STRING,
  temperature DOUBLE,
  humidity DOUBLE,
  wind_speed DOUBLE,
  pressure DOUBLE,
  weather_condition STRING,
  condition STRING,
  rainfall DOUBLE,
  uv_index DOUBLE,
  visibility DOUBLE,
  feels_like_temperature STRING,
  station_id STRING,
  data_provider STRING,
  sensor_payload STRING,
  _rescued_data STRING,
  ingestion_time TIMESTAMP,
  source_file STRING,
  batch_id STRING,
  load_date DATE,
  record_status STRING
)
USING DELTA
PARTITIONED BY (load_date)
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
);

In [0]:
#  Configure stream with safe JSON parsing

sensor_schema = StructType([
    StructField("temperature",       StringType(), True),
    StructField("humidity",          StringType(), True),
    StructField("wind_speed",        StringType(), True),
    StructField("pressure",          StringType(), True),
    StructField("weather_condition", StringType(), True),
    StructField("rainfall",          StringType(), True),
    StructField("uv_index",          StringType(), True),
    StructField("visibility",        StringType(), True),
])

raw_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC)
        .option("header", True)
        .option("maxFilesPerTrigger", 1)
        .load(RAW_PATH)
)

# Parse sensor_payload using SQL expr — works at execution time not definition time
# from_json returns all NULLs when sensor_payload is NULL (Files 01-12)
# So this is safe for ALL files, not just File 13
parsed_stream = raw_stream \
    .withColumn("sensor_struct",
        F.from_json(F.col("sensor_payload"), sensor_schema)) \
    .withColumn("temperature",
        F.coalesce(
            F.expr("try_cast(temperature as double)"),
            F.expr("try_cast(sensor_struct.temperature as double)")
        )) \
    .withColumn("humidity",
        F.coalesce(
            F.expr("try_cast(humidity as double)"),
            F.expr("try_cast(sensor_struct.humidity as double)")
        )) \
    .withColumn("wind_speed",
        F.coalesce(
            F.expr("try_cast(wind_speed as double)"),
            F.expr("try_cast(sensor_struct.wind_speed as double)")
        )) \
    .withColumn("pressure",
        F.coalesce(
            F.expr("try_cast(pressure as double)"),
            F.expr("try_cast(sensor_struct.pressure as double)")
        )) \
    .withColumn("rainfall",
        F.coalesce(
            F.expr("try_cast(rainfall as double)"),
            F.expr("try_cast(sensor_struct.rainfall as double)")
        )) \
    .withColumn("uv_index",
        F.coalesce(
            F.expr("try_cast(uv_index as double)"),
            F.expr("try_cast(sensor_struct.uv_index as double)")
        )) \
    .withColumn("visibility",
        F.coalesce(
            F.expr("try_cast(visibility as double)"),
            F.expr("try_cast(sensor_struct.visibility as double)")
        )) \
    .withColumn("weather_condition",
        F.coalesce(
            F.col("weather_condition"),
            F.col("sensor_struct.weather_condition")
        )) \
    .drop("sensor_struct")

print("Stream configured successfully.")
print("Schema:")
parsed_stream.printSchema()

Stream configured successfully.
Schema:
root
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- rainfall: double (nullable = true)
 |-- station_id: string (nullable = true)
 |-- data_provider: string (nullable = true)
 |-- uv_index: double (nullable = true)
 |-- visibility: double (nullable = true)
 |-- condition: string (nullable = true)
 |-- feels_like_temperature: string (nullable = true)
 |-- sensor_payload: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
#metadata
bronze_ready = (
    parsed_stream
        .withColumn("ingestion_time", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("batch_id", F.col("_metadata.file_modification_time").cast("string"))
        .withColumn("load_date", F.current_date())
        .withColumn("record_status", F.lit("raw"))
)

In [0]:
# for q in spark.streams.active:
#     q.stop()

In [0]:
# Start stream and WAIT for it to finish

bronze_query = (
    bronze_ready.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

bronze_query.awaitTermination()  
print("Bronze stream complete!")

Bronze stream started — waiting for completion...
Bronze stream complete!


In [0]:
display(spark.sql(f"SELECT source_file, COUNT(*) cnt FROM {BRONZE_TABLE} GROUP BY source_file ORDER BY source_file"))

source_file,cnt
/Volumes/weather_project/default/raw/weather_01.csv,288
/Volumes/weather_project/default/raw/weather_02.csv,144
/Volumes/weather_project/default/raw/weather_03.csv,144
/Volumes/weather_project/default/raw/weather_04.csv,144
/Volumes/weather_project/default/raw/weather_05.csv,36
/Volumes/weather_project/default/raw/weather_06.csv,72
/Volumes/weather_project/default/raw/weather_07.csv,72
/Volumes/weather_project/default/raw/weather_08.csv,72
/Volumes/weather_project/default/raw/weather_09.csv,144
/Volumes/weather_project/default/raw/weather_10.csv,288


In [0]:
spark.read.table(BRONZE_TABLE).display()

city,country,event_time,temperature,humidity,wind_speed,pressure,weather_condition,condition,rainfall,uv_index,visibility,feels_like_temperature,station_id,data_provider,sensor_payload,_rescued_data,ingestion_time,source_file,batch_id,load_date,record_status
Chennai,India,2026-03-01 00:00:00,40.1,69.0,10.8,1005.7,Clear,null,38.0,null,null,null,STN_CHN_01,WeatherPro,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
Mumbai,India,2026-03-01 00:00:00,39.9,73.0,10.0,1013.7,Cloudy,null,20.3,null,null,null,STN_MUM_02,AtmosAPI,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
Delhi,India,2026-03-01 00:00:00,38.3,44.0,15.3,1002.8,Foggy,null,28.2,null,null,null,STN_DEL_01,ClimaData,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
Chennai,India,2026-03-01 01:00:00,41.6,77.0,11.9,1006.9,Foggy,null,28.8,null,null,null,STN_CHN_01,AtmosAPI,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
Mumbai,India,2026-03-01 01:00:00,39.7,69.0,16.4,1012.6,Clear,null,23.7,null,null,null,STN_MUM_01,ClimaData,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
Delhi,India,2026-03-01 01:00:00,38.4,41.0,12.0,1009.2,Foggy,null,39.5,null,null,null,STN_DEL_01,WeatherPro,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
Chennai,India,2026-03-01 02:00:00,39.9,72.0,9.6,1005.5,Cloudy,null,21.6,null,null,null,STN_CHN_01,AtmosAPI,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
Mumbai,India,2026-03-01 02:00:00,36.0,72.0,20.2,1008.8,Thunderstorm,null,25.0,null,null,null,STN_MUM_02,SkyWatch,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
Delhi,India,2026-03-01 02:00:00,37.1,38.0,15.3,1003.8,Cloudy,null,42.8,null,null,null,STN_DEL_01,AtmosAPI,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
Chennai,India,2026-03-01 03:00:00,39.7,70.0,17.0,1006.5,Thunderstorm,null,34.5,null,null,null,STN_CHN_01,WeatherPro,null,null,2026-04-16T10:18:49.092Z,/Volumes/weather_project/default/raw/weather_05.csv,2026-04-10 09:22:40,2026-04-16,raw
